# BDD100K Dataset Download and Setup

This notebook downloads the BDD100K dataset (10k images) and prepares it for use in computer vision tasks.

In [ ]:
ROOT_DIR: str = "./"

# To save models on gogole drive and make them easy to get back
# from google.colab import drive
# drive.mount('/content/drive')

# MODEL_DIR: str = "./drive/MyDrive/models/"
# To locally save on the google collab instance
MODEL_DIR: str = "./models/"

: 

In [ ]:
# Detect available GPUs using TensorFlow
# All comments in English as requested.

from __future__ import annotations

import tensorflow as tf

def list_available_gpus() -> list[str]:
    """Return a list of available GPU device names."""
    gpus = tf.config.list_physical_devices("GPU")
    return [gpu.name for gpu in gpus]

gpus = list_available_gpus()
if gpus:
    print("Available GPUs:")
    for gpu in gpus:
        print(f" - {gpu}")
else:
    print("No GPU detected.")


# 0. Installing dependencies

In [ ]:
%pip install pillow numpy opencv-python matplotlib tqdm opencv-python-headless albumentations scikit-image

## 1. Fetch Dataset

Download the BDD100K images dataset (10k version) from the official source.

In [ ]:
from src.fetch_dataset import fetch_dataset

# Dataset configuration
DATASET_NAME: str = "bdd100k_images_10k"
DATASET_URL = f"http://128.32.162.150/bdd100k/{DATASET_NAME}.zip"
DATASET_FILE = f"{ROOT_DIR}/{DATASET_NAME}.zip"
OUTPUT_DIR = f"{ROOT_DIR}/{DATASET_NAME}"

fetch_dataset(DATASET_URL, DATASET_FILE, OUTPUT_DIR)

## 1.1 Display Sample Images

Load and display a few sample images from the dataset.

In [ ]:

from src.fetch_dataset import display_sample_images

display_sample_images(OUTPUT_DIR)


---

# 2. Data Augmentation for GAN-based Image Reconstruction

For training a robust GAN model for image reconstruction, we need to create augmented versions of our dataset with various degradations. The model will learn to reconstruct clean images from these degraded inputs.

## 2.1 Test Augmentation Classes on Sample Images

Visualize different augmentation techniques on sample images.

In [ ]:
from PIL import Image

import os
import glob

from src.augmentation import GroundTruthAugmentor, ImageDegradator

all_images: list[str] = glob.glob(os.path.join(OUTPUT_DIR, '**', '*.jpg'), recursive=True)

# Create instances
augmentor = GroundTruthAugmentor()
degradator = ImageDegradator()
print("GroundTruthAugmentor and ImageDegradator classes created!")
print("- GroundTruthAugmentor: Handles stretch, rotation, zoom, and colorimetry")
print("- ImageDegradator: Handles noise, blur, occlusion, scratches, and rain drops")

augmentor.demo(Image.open(all_images[0]))
degradator.demo(Image.open(all_images[0]))

## 2.2 Generate Augmented Dataset

Generate augmented image pairs (degraded, clean) for GAN training.

In [ ]:
from src.augmentation import generate_dataset

# Configuration
AUGMENTED_DIR = f"{ROOT_DIR}/data_augmented"

# Create augmented dataset using parallel processing on all CPU cores
clean_dir, degraded_dir, total_generated = generate_dataset(
    all_images,
    AUGMENTED_DIR,
    num_augmentations_per_image=1,
    num_degradations_per_image=2,
    force_regenerate=False,
    n_jobs=None  # None = use all available CPU cores
)

## 2.2.1 Visualize Augmented Image Pairs

Display some examples of degraded/clean image pairs from the augmented dataset.

In [ ]:
import matplotlib.pyplot as plt

# Get sample augmented images
degraded_images = sorted(glob.glob(os.path.join(degraded_dir, '*.jpg')))
clean_images = sorted(glob.glob(os.path.join(clean_dir, '*.jpg')))

if len(degraded_images) > 0:
    # Display 4 pairs
    num_pairs = min(4, len(degraded_images))

    fig, axes = plt.subplots(num_pairs, 2, figsize=(12, num_pairs * 3))
    if num_pairs == 1:
        axes = axes.reshape(1, -1)

    for i in range(num_pairs):
        # Load degraded (input)
        degraded_img = Image.open(degraded_images[i])
        axes[i, 0].imshow(degraded_img)
        axes[i, 0].set_title(f'Degraded (Input) - {os.path.basename(degraded_images[i])}',
                            fontsize=10)
        axes[i, 0].axis('off')

        # Load clean (target)
        clean_img = Image.open(clean_images[i])
        axes[i, 1].imshow(clean_img)
        axes[i, 1].set_title(f'Clean (Target) - {os.path.basename(clean_images[i])}',
                            fontsize=10)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

    print(f"✓ Displayed {num_pairs} image pairs from augmented dataset")
    print(f"\nDataset Summary:")
    print(f"  - Total degraded images: {len(degraded_images)}")
    print(f"  - Total clean images: {len(clean_images)}")
    print(f"  - Ready for GAN training!")
else:
    print("No augmented images found")

---

# GAN-based Image Reconstruction Model

Build a Generative Adversarial Network (GAN) to reconstruct clean images from degraded inputs.

## 3. Install TensorFlow and Setup

Install TensorFlow and required dependencies for building the GAN model.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow.keras.backend as K

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Set memory growth for GPU (if available)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

print("✓ TensorFlow imported successfully")

## 4. Data Pipeline Setup

Create TensorFlow data pipeline for loading and preprocessing image pairs.

In [ ]:
from src.create_dataset import create_dataset

# Configuration
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 8
BUFFER_SIZE = 1000

# Create train dataset
train_dataset = create_dataset(degraded_dir, clean_dir, IMG_HEIGHT, IMG_WIDTH, BATCH_SIZE, BUFFER_SIZE, shuffle=True)

print(f"✓ Dataset created successfully")
print(f"  - Image size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Data range: [-1, 1]")

## 5. Build Generator (U-Net Architecture)

Create the Generator model using U-Net architecture for image reconstruction.

In [ ]:
def downsample(filters, size, apply_batchnorm=True):
    """Downsampling block for U-Net encoder"""
    initializer = tf.random_normal_initializer(0., 0.02)

    result = keras.Sequential()
    result.add(layers.Conv2D(filters, size, strides=2, padding='same',
                            kernel_initializer=initializer, use_bias=False))

    if apply_batchnorm:
        result.add(layers.BatchNormalization())

    result.add(layers.LeakyReLU())

    return result

def upsample(filters, size, apply_dropout=False):
    """Upsampling block for U-Net decoder"""
    initializer = tf.random_normal_initializer(0., 0.02)

    result = keras.Sequential()
    result.add(layers.Conv2DTranspose(filters, size, strides=2, padding='same',
                                     kernel_initializer=initializer, use_bias=False))
    result.add(layers.BatchNormalization())

    if apply_dropout:
        result.add(layers.Dropout(0.5))

    result.add(layers.ReLU())

    return result

def build_generator():
    """
    Build U-Net Generator for image reconstruction

    Architecture:
    - Encoder: Downsamples degraded image to bottleneck
    - Decoder: Upsamples with skip connections to reconstruct clean image
    """
    inputs = layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3])

    # Encoder (downsampling)
    down_stack = [
        downsample(64, 4, apply_batchnorm=False),  # (batch, 128, 128, 64)
        downsample(128, 4),                         # (batch, 64, 64, 128)
        downsample(256, 4),                         # (batch, 32, 32, 256)
        downsample(512, 4),                         # (batch, 16, 16, 512)
        downsample(512, 4),                         # (batch, 8, 8, 512)
        downsample(512, 4),                         # (batch, 4, 4, 512)
        downsample(512, 4),                         # (batch, 2, 2, 512)
        downsample(512, 4),                         # (batch, 1, 1, 512) - Bottleneck
    ]

    # Decoder (upsampling with skip connections)
    up_stack = [
        upsample(512, 4, apply_dropout=True),       # (batch, 2, 2, 1024)
        upsample(512, 4, apply_dropout=True),       # (batch, 4, 4, 1024)
        upsample(512, 4, apply_dropout=True),       # (batch, 8, 8, 1024)
        upsample(512, 4),                           # (batch, 16, 16, 1024)
        upsample(256, 4),                           # (batch, 32, 32, 512)
        upsample(128, 4),                           # (batch, 64, 64, 256)
        upsample(64, 4),                            # (batch, 128, 128, 128)
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4, strides=2, padding='same',
                                 kernel_initializer=initializer,
                                 activation='tanh')  # (batch, 256, 256, 3)

    x = inputs

    # Downsampling through the model
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])

    # Upsampling and establishing skip connections
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    x = last(x)

    return keras.Model(inputs=inputs, outputs=x, name='Generator')

print("✓ Generator build function defined")
print("  Note: Use the 'Load Pre-trained Models' cell below to create or load the generator")

## 6. Build Discriminator (PatchGAN)

Create the Discriminator model to distinguish real images from reconstructed ones.

In [ ]:
def build_discriminator():
    """
    Build PatchGAN Discriminator

    The discriminator classifies whether 30x30 patches are real or fake.
    This encourages high-frequency details in the generated images.

    Input: Concatenated degraded image and target/generated image
    Output: Patch-based predictions (real/fake for each patch)
    """
    initializer = tf.random_normal_initializer(0., 0.02)

    # Input is degraded image
    inp = layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3], name='degraded_image')

    # Target is either clean (real) or generated (fake) image
    tar = layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3], name='target_image')

    # Concatenate inputs
    x = layers.Concatenate()([inp, tar])  # (batch, 256, 256, 6)

    # Downsampling layers
    down1 = downsample(64, 4, False)(x)      # (batch, 128, 128, 64)
    down2 = downsample(128, 4)(down1)         # (batch, 64, 64, 128)
    down3 = downsample(256, 4)(down2)         # (batch, 32, 32, 256)

    # Zero padding
    zero_pad1 = layers.ZeroPadding2D()(down3)  # (batch, 34, 34, 256)
    conv = layers.Conv2D(512, 4, strides=1,
                        kernel_initializer=initializer,
                        use_bias=False)(zero_pad1)  # (batch, 31, 31, 512)

    batchnorm1 = layers.BatchNormalization()(conv)
    leaky_relu = layers.LeakyReLU()(batchnorm1)

    zero_pad2 = layers.ZeroPadding2D()(leaky_relu)  # (batch, 33, 33, 512)

    # Output layer - single channel for real/fake classification
    last = layers.Conv2D(1, 4, strides=1,
                        kernel_initializer=initializer)(zero_pad2)  # (batch, 30, 30, 1)

    return keras.Model(inputs=[inp, tar], outputs=last, name='Discriminator')

print("✓ Discriminator build function defined")
print("  Note: Use the 'Load Pre-trained Models' cell to create or load the discriminator")

## 7.0 Load or Build Models

**Configure Model Creation:**

Set `LOAD_PRETRAINED_MODELS = True` to load saved models, or `False` to build from scratch.

### 📚 Model Loading Guide

**Three Ways to Work with Models:**

#### **Option 1: Build Fresh Models (Default)**
- Set `LOAD_PRETRAINED_MODELS = False`
- Models are created from scratch using build functions
- Use this for starting new training

#### **Option 2: Load Saved Models**
- Set `LOAD_PRETRAINED_MODELS = True`
- Specify paths to `.keras` or `.h5` files
- Only loads model weights (not optimizer state)
- Use this for:
  - Inference on new data
  - Starting fresh training with pre-trained weights
  - Switching between different trained models

#### **Option 3: Restore from Checkpoint**
- Use the checkpoint restoration cell (Section 18.0.1)
- Restores model weights AND optimizer state
- Use this for:
  - Resuming interrupted training
  - Continuing training from exact same point

---

In [ ]:
import os

# Configuration for loading models
LOAD_PRETRAINED_MODELS = True  # Set to True to load saved models
GENERATOR_MODEL_PATH: str = f'{MODEL_DIR}/generator.keras'
DISCRIMINATOR_MODEL_PATH: str = f'{MODEL_DIR}/discriminator.keras'

# Save models
SAVE_PATH: str = f'{MODEL_DIR}/new/'
os.makedirs(SAVE_PATH, exist_ok=True)
GENERATOR_SAVE_PATH = f'{SAVE_PATH}/generator.keras'
DISCRIMINATOR_SAVE_PATH = f'{SAVE_PATH}/discriminator.keras'


# Save in TensorFlow SavedModel format
def save_models(generator, discriminator):
    generator.save(GENERATOR_SAVE_PATH)
    discriminator.save(DISCRIMINATOR_SAVE_PATH)

    print("✓ Models saved successfully")
    print(f"  - Generator: {GENERATOR_SAVE_PATH}")
    print(f"  - Discriminator: {DISCRIMINATOR_SAVE_PATH}")


def load_or_build_models(load_pretrained=False, generator_path=None, discriminator_path=None):
    """
    Load pre-trained models or build new ones

    Args:
        load_pretrained: If True, load models from saved files
        generator_path: Path to saved generator model
        discriminator_path: Path to saved discriminator model

    Returns:
        Tuple of (generator, discriminator)
    """
    if load_pretrained and generator_path and discriminator_path:
        if os.path.exists(generator_path) and os.path.exists(discriminator_path):
            print("Loading pre-trained models...")
            try:
                gen = keras.models.load_model(generator_path, compile=False)
                disc = keras.models.load_model(discriminator_path, compile=False)

                print("✓ Models loaded successfully!")
                print(f"  - Generator loaded from: {generator_path}")
                print(f"  - Generator parameters: {gen.count_params():,}")
                print(f"  - Discriminator loaded from: {discriminator_path}")
                print(f"  - Discriminator parameters: {disc.count_params():,}")

                return gen, disc
            except Exception as e:
                print(f"⚠ Error loading models: {e}")
                print("Building new models instead...")
        else:
            print(f"⚠ Model files not found:")
            if not os.path.exists(generator_path):
                print(f"  - Generator: {generator_path} (not found)")
            if not os.path.exists(discriminator_path):
                print(f"  - Discriminator: {discriminator_path} (not found)")
            print("Building new models instead...")

    # Build new models
    print("Building new models from scratch...")
    gen = build_generator()
    disc = build_discriminator()

    print("✓ New models built successfully")
    print(f"  - Generator parameters: {gen.count_params():,}")
    print(f"  - Discriminator parameters: {disc.count_params():,}")

    return gen, disc

# Load or build models based on configuration
generator, discriminator = load_or_build_models(
    load_pretrained=LOAD_PRETRAINED_MODELS,
    generator_path=GENERATOR_MODEL_PATH,
    discriminator_path=DISCRIMINATOR_MODEL_PATH
)

print("\n" + "="*60)
if LOAD_PRETRAINED_MODELS:
    print("MODE: Loading pre-trained models")
else:
    print("MODE: Building new models from scratch")
print("="*60)

## 7. Define Loss Functions

Define the loss functions for training the GAN.

## 7.1 Improved Loss Configuration

**Key Improvements:**
1. **Increased LAMBDA to 200-300**: Higher weight on L1 loss for better pixel-wise reconstruction
2. **Added Perceptual Loss**: Use VGG features for semantic similarity
3. **Multi-scale Loss**: Evaluate reconstruction at multiple resolutions

In [ ]:
# BUILD PERCEPTUAL LOSS MODEL (VGG-based)
# This helps the model focus on semantic features rather than just pixel differences

from tensorflow.keras.applications import VGG19

def build_perceptual_model():
    """
    Build VGG19-based perceptual loss model
    Uses intermediate layers for feature extraction
    """
    vgg = VGG19(include_top=False, weights='imagenet', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

    # Use multiple layers for multi-scale feature matching
    layer_names = ['block1_conv2', 'block2_conv2', 'block3_conv4', 'block4_conv4', 'block5_conv4']
    outputs = [vgg.get_layer(name).output for name in layer_names]

    model = keras.Model(inputs=vgg.input, outputs=outputs)
    model.trainable = False  # Freeze VGG weights

    return model

# Build perceptual model
perceptual_model = build_perceptual_model()

print("✓ Perceptual loss model built")
print("  - Using VGG19 features from 5 layers")
print("  - Model frozen (not trainable)")

In [ ]:
# IMPROVED LOSS FUNCTIONS WITH PERCEPTUAL LOSS

# Loss object
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# Updated Lambda values for better reconstruction
LAMBDA_L1 = 200  # Increased from 100 to 200 for stronger pixel-wise guidance
LAMBDA_PERCEPTUAL = 10  # Weight for perceptual loss

def preprocess_for_vgg(images):
    """
    Preprocess images for VGG19 (convert from [-1,1] to [0,255] and apply VGG preprocessing)
    """
    # Convert from [-1, 1] to [0, 255]
    images = (images + 1) * 127.5
    # Apply VGG preprocessing (RGB to BGR and zero-center)
    return tf.keras.applications.vgg19.preprocess_input(images)

def perceptual_loss(generated_images, target_images):
    """
    Calculate perceptual loss using VGG19 features
    Compares high-level features rather than raw pixels
    """
    # Preprocess for VGG
    gen_preprocessed = preprocess_for_vgg(generated_images)
    target_preprocessed = preprocess_for_vgg(target_images)

    # Extract features
    gen_features = perceptual_model(gen_preprocessed)
    target_features = perceptual_model(target_preprocessed)

    # Calculate loss across all feature layers
    loss = 0
    for gen_feat, target_feat in zip(gen_features, target_features):
        loss += tf.reduce_mean(tf.abs(gen_feat - target_feat))

    return loss / len(gen_features)

def discriminator_loss(real_output, fake_output):
    """
    Discriminator loss function

    Real images should be classified as 1 (real)
    Fake images should be classified as 0 (fake)
    """
    real_loss = loss_object(tf.ones_like(real_output), real_output)
    fake_loss = loss_object(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output, generated_image, target_image):
    """
    IMPROVED Generator loss function

    Combines:
    1. Adversarial loss: Fool discriminator (fake images classified as real)
    2. L1 loss: Pixel-wise similarity to target image (INCREASED WEIGHT)
    3. Perceptual loss: High-level feature similarity (NEW)

    The combination ensures both pixel-accurate and semantically correct reconstructions.
    """
    # 1. Adversarial loss - generator wants discriminator to classify fakes as real
    gan_loss = loss_object(tf.ones_like(fake_output), fake_output)

    # 2. L1 loss - mean absolute error between generated and target
    l1_loss = tf.reduce_mean(tf.abs(target_image - generated_image))

    # 3. Perceptual loss - feature-based similarity (NEW)
    percep_loss = perceptual_loss(generated_image, target_image)

    # Total generator loss with weighted components
    total_gen_loss = gan_loss + (LAMBDA_L1 * l1_loss) + (LAMBDA_PERCEPTUAL * percep_loss)

    return total_gen_loss, gan_loss, l1_loss, percep_loss

print("✓ IMPROVED Loss functions defined")
print(f"  - Discriminator loss: Binary Cross-Entropy")
print(f"  - Generator loss components:")
print(f"    1. Adversarial loss (GAN)")
print(f"    2. L1 loss (pixel-wise) × {LAMBDA_L1} (INCREASED from 100)")
print(f"    3. Perceptual loss (VGG features) × {LAMBDA_PERCEPTUAL} (NEW)")
print(f"\n  Total generator loss = GAN_loss + {LAMBDA_L1}×L1_loss + {LAMBDA_PERCEPTUAL}×Perceptual_loss")

## 8. Setup Optimizers and Checkpoints

Configure optimizers and checkpointing for model training.

In [ ]:
# Optimizers
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# Checkpoint directory
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator
)

print("✓ Optimizers configured")
print(f"  - Generator optimizer: Adam (lr=2e-4, beta_1=0.5)")
print(f"  - Discriminator optimizer: Adam (lr=2e-4, beta_1=0.5)")
print(f"  - Checkpoint directory: {checkpoint_dir}")

## 8.0.1 Restore from Checkpoint (Optional)

**Note:** This is different from loading saved models!
- **Checkpoints**: Include optimizer state - use for resuming interrupted training
- **Saved Models**: Only model weights - use for inference or starting fresh training

Uncomment and run the cell below to restore from a checkpoint.

In [ ]:
# Restore from checkpoint (uncomment to use)
# This will restore both model weights AND optimizer state

# Check if checkpoint exists
# checkpoint_status = checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))
# if tf.train.latest_checkpoint(checkpoint_dir):
#     print(f"✓ Restored from checkpoint: {tf.train.latest_checkpoint(checkpoint_dir)}")
#     print("  - Model weights restored")
#     print("  - Optimizer state restored")
#     print("  - Ready to resume training from where it left off")
# else:
#     print("ℹ No checkpoint found - starting fresh training")

print("ℹ Checkpoint restoration is commented out")
print("  Uncomment the code above to restore from checkpoint")

## 8.1 Alternative Optimizer Configurations

**Additional Optimization Strategies to Try:**

If the model still struggles, try these optimizer adjustments:

1. **Lower Learning Rate with Warmup**
   - Start: `1e-4` or `5e-5`
   - Gradually increase over first 10 epochs
   
2. **Separate Learning Rates**
   - Generator: `1e-4` (slower)
   - Discriminator: `2e-4` (current)
   
3. **Learning Rate Scheduling**
   - Cosine decay
   - Exponential decay after plateau
   
4. **Gradient Clipping**
   - Prevents exploding gradients
   - Clip norm: 1.0

In [ ]:
# OPTION 1: IMPROVED OPTIMIZERS WITH LOWER LEARNING RATES
# Uncomment this section to use improved optimizer settings

# Lower learning rates can provide more stable training
# generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
# discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# OPTION 2: WITH LEARNING RATE SCHEDULE
# Gradually decrease learning rate over time

def create_lr_schedule(initial_lr=2e-4, decay_steps=1000, decay_rate=0.96):
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=initial_lr,
        decay_steps=decay_steps,
        decay_rate=decay_rate,
        staircase=True
    )
    return lr_schedule

gen_lr_schedule = create_lr_schedule(initial_lr=1e-3)
disc_lr_schedule = create_lr_schedule(initial_lr=2e-4)

generator_optimizer = tf.keras.optimizers.Adam(gen_lr_schedule, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(disc_lr_schedule, beta_1=0.5)

print("✓ Alternative optimizer configurations available (commented out)")
# print("  Uncomment to use lower learning rates or learning rate schedules")

## 9. Training Step Function

Define the training step for the GAN.

In [ ]:
@tf.function
def train_step(degraded_images, target_images):
    """
    IMPROVED Single training step for the GAN with perceptual loss

    Args:
        degraded_images: Batch of degraded input images
        target_images: Batch of clean target images

    Returns:
        Dictionary of losses for logging (now includes perceptual loss)
    """
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generate fake images
        generated_images = generator(degraded_images, training=True)

        # Discriminator predictions
        real_output = discriminator([degraded_images, target_images], training=True)
        fake_output = discriminator([degraded_images, generated_images], training=True)

        # Calculate losses (now includes perceptual loss)
        gen_total_loss, gen_gan_loss, gen_l1_loss, gen_percep_loss = generator_loss(
            fake_output, generated_images, target_images
        )
        disc_loss = discriminator_loss(real_output, fake_output)

    # Calculate gradients
    generator_gradients = gen_tape.gradient(
        gen_total_loss, generator.trainable_variables
    )
    discriminator_gradients = disc_tape.gradient(
        disc_loss, discriminator.trainable_variables
    )

    # Apply gradients
    generator_optimizer.apply_gradients(
        zip(generator_gradients, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(discriminator_gradients, discriminator.trainable_variables)
    )

    return {
        'gen_total_loss': gen_total_loss,
        'gen_gan_loss': gen_gan_loss,
        'gen_l1_loss': gen_l1_loss,
        'gen_percep_loss': gen_percep_loss,  # NEW: Track perceptual loss
        'disc_loss': disc_loss
    }

print("✓ IMPROVED Training step function defined")
print("  - Now includes perceptual loss tracking")

## 10.1. Visualization Helper Functions

Create functions to visualize training progress.

In [ ]:
def generate_images(model, test_degraded, test_target, save_path=None):
    """
    Generate and display reconstructed images

    Args:
        model: Generator model
        test_degraded: Degraded input images
        test_target: Clean target images
        save_path: Optional path to save the figure
    """
    prediction = model(test_degraded, training=False)

    # Convert from [-1, 1] to [0, 1] for display
    test_degraded_display = (test_degraded + 1) / 2.0
    test_target_display = (test_target + 1) / 2.0
    prediction_display = (prediction + 1) / 2.0

    # Display up to 3 examples
    num_examples = min(3, test_degraded.shape[0])

    fig, axes = plt.subplots(num_examples, 3, figsize=(15, num_examples * 5))

    if num_examples == 1:
        axes = axes.reshape(1, -1)

    for i in range(num_examples):
        # Degraded input
        axes[i, 0].imshow(test_degraded_display[i])
        axes[i, 0].set_title('Degraded Input', fontsize=14, fontweight='bold')
        axes[i, 0].axis('off')

        # Generated reconstruction
        axes[i, 1].imshow(prediction_display[i])
        axes[i, 1].set_title('Generated Reconstruction', fontsize=14, fontweight='bold')
        axes[i, 1].axis('off')

        # Ground truth clean
        axes[i, 2].imshow(test_target_display[i])
        axes[i, 2].set_title('Ground Truth (Clean)', fontsize=14, fontweight='bold')
        axes[i, 2].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f"✓ Saved visualization to {save_path}")

    plt.show()

# Get a test batch for visualization
test_degraded, test_target = next(iter(train_dataset))

print("✓ Visualization functions defined")
print(f"  - Test batch size: {test_degraded.shape[0]}")

## 10.1 Visualize Model Reconstruction

Test the model's reconstruction capability and visualize results in a clear comparison format.

In [ ]:
def visualize_reconstruction_comparison(model, degraded_images, original_images, num_samples=4):
    """
    Visualize model reconstruction in a clear tabular format

    Layout: Degraded | Reconstructed | Original

    Args:
        model: Generator model
        degraded_images: Degraded input images (normalized to [-1, 1])
        original_images: Original clean images (normalized to [-1, 1])
        num_samples: Number of samples to display
    """
    # Generate reconstructions
    reconstructed_images = model(degraded_images, training=False)

    # Convert from [-1, 1] to [0, 1] for display
    degraded_display = (degraded_images + 1) / 2.0
    reconstructed_display = (reconstructed_images + 1) / 2.0
    original_display = (original_images + 1) / 2.0

    # Limit number of samples
    num_samples = min(num_samples, degraded_images.shape[0])

    # Create figure with 3 columns: Degraded | Reconstructed | Original
    fig, axes = plt.subplots(num_samples, 3, figsize=(18, num_samples * 6))

    # Handle single row case
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    # Column titles
    column_titles = ['Degraded Image', 'Reconstructed Image', 'Original Image']

    for i in range(num_samples):
        for j, title in enumerate(column_titles):
            # Select the appropriate image
            if j == 0:
                img = degraded_display[i]
            elif j == 1:
                img = reconstructed_display[i]
            else:
                img = original_display[i]

            # Display image
            axes[i, j].imshow(img)

            # Add title only to first row
            if i == 0:
                axes[i, j].set_title(title, fontsize=16, fontweight='bold', pad=20)

            # Add row label on the left
            if j == 0:
                axes[i, j].set_ylabel(f'Sample {i+1}', fontsize=14, fontweight='bold', rotation=0,
                                     labelpad=60, va='center')

            axes[i, j].axis('off')

    plt.tight_layout()
    plt.subplots_adjust(wspace=0.05, hspace=0.1)

    # Save figure
    output_path = 'reconstruction_comparison.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"✓ Saved comparison visualization to {output_path}")

    plt.show()

    return reconstructed_display

# Test the visualization with current (untrained) model
print("Testing model reconstruction visualization...")
print("Note: Model is untrained, so reconstructions will be poor initially.\n")

# Get test samples
test_batch_degraded, test_batch_original = next(iter(train_dataset))

# Visualize reconstruction comparison
reconstructed = visualize_reconstruction_comparison(
    generator,
    test_batch_degraded,
    test_batch_original,
    num_samples=4
)

print("\n✓ Reconstruction comparison displayed")
print("  Format: Degraded → Reconstructed → Original")
print("  This visualization will improve as training progresses!")

## 11. Training Loop

Train the GAN model.

In [ ]:
import time
from IPython.display import clear_output
import numpy as np

def train(dataset, epochs, checkpoint_interval=5):
    """
    IMPROVED Train the GAN model with perceptual loss tracking

    Args:
        dataset: Training dataset
        epochs: Number of epochs to train
        checkpoint_interval: Save checkpoint every N epochs
    """
    # Create output directory for samples
    sample_dir = './training_samples'
    os.makedirs(sample_dir, exist_ok=True)

    # Training history (now includes perceptual loss)
    history = {
        'gen_total_loss': [],
        'gen_gan_loss': [],
        'gen_l1_loss': [],
        'gen_percep_loss': [],  # NEW
        'disc_loss': []
    }

    for epoch in range(epochs):
        start_time = time.time()

        # Training metrics for this epoch
        epoch_gen_total_loss = []
        epoch_gen_gan_loss = []
        epoch_gen_l1_loss = []
        epoch_gen_percep_loss = []  # NEW
        epoch_disc_loss = []

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 60)
        last_time: float = time.time()

        # Train on all batches
        for step, (degraded_images, target_images) in enumerate(dataset):
            losses = train_step(degraded_images, target_images)

            epoch_gen_total_loss.append(losses['gen_total_loss'].numpy())
            epoch_gen_gan_loss.append(losses['gen_gan_loss'].numpy())
            epoch_gen_l1_loss.append(losses['gen_l1_loss'].numpy())
            epoch_gen_percep_loss.append(losses['gen_percep_loss'].numpy())  # NEW
            epoch_disc_loss.append(losses['disc_loss'].numpy())

            # Print progress every 10 steps
            if step % 10 == 0 or last_time + 20 < time.time():
                last_time = time.time()
                elapsed = time.time() - start_time
                print(f"  Step {step}/{len(dataset)} ({(step / ((elapsed or 1) / 60))} step/min | Epoch ETA: {elapsed / (step or 1) * (len(dataset) - step)}): "
                      f"Gen Loss: {losses['gen_total_loss']:.4f} "
                      f"(GAN: {losses['gen_gan_loss']:.4f}, "
                      f"L1: {losses['gen_l1_loss']:.4f}, "
                      f"Percep: {losses['gen_percep_loss']:.4f}), "
                      f"Disc Loss: {losses['disc_loss']:.4f}")

        # Calculate average losses for this epoch
        avg_gen_total = np.mean(epoch_gen_total_loss)
        avg_gen_gan = np.mean(epoch_gen_gan_loss)
        avg_gen_l1 = np.mean(epoch_gen_l1_loss)
        avg_gen_percep = np.mean(epoch_gen_percep_loss)  # NEW
        avg_disc = np.mean(epoch_disc_loss)

        history['gen_total_loss'].append(avg_gen_total)
        history['gen_gan_loss'].append(avg_gen_gan)
        history['gen_l1_loss'].append(avg_gen_l1)
        history['gen_percep_loss'].append(avg_gen_percep)  # NEW
        history['disc_loss'].append(avg_disc)

        epoch_time = time.time() - start_time

        print(f"\n  Epoch {epoch + 1} Summary:")
        print(f"    Generator Total Loss: {avg_gen_total:.4f}")
        print(f"    ├─ GAN Loss: {avg_gen_gan:.4f}")
        print(f"    ├─ L1 Loss: {avg_gen_l1:.4f}")
        print(f"    └─ Perceptual Loss: {avg_gen_percep:.4f} (NEW)")
        print(f"    Discriminator Loss: {avg_disc:.4f}")
        print(f"    Time: {epoch_time:.2f}s")

        save_models(generator, discriminator)

        # Generate sample images
        sample_path = os.path.join(sample_dir, f'epoch_{epoch + 1:03d}.png')
        generate_images(generator, test_degraded, test_target, save_path=sample_path)

        # Save checkpoint
        if (epoch + 1) % checkpoint_interval == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f"  ✓ Checkpoint saved at epoch {epoch + 1}")

    # Final checkpoint
    checkpoint.save(file_prefix=checkpoint_prefix)
    print(f"\n✓ Training complete! Final checkpoint saved.")

    return history

print("✓ IMPROVED Training function defined with perceptual loss tracking")
print("  Ready to start training with better loss formulation!")

## 12. Start Training

Execute the training process.

In [ ]:
# Training configuration
EPOCHS = 50  # Adjust based on your needs and computational resources
CHECKPOINT_INTERVAL = 5  # Save checkpoint every 5 epochs

print("Training Configuration:")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Image size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"  - Checkpoint interval: {CHECKPOINT_INTERVAL} epochs")
print(f"\n{'='*60}")
print("Starting training...")
print(f"{'='*60}\n")

# Train the model
history = train(train_dataset, epochs=EPOCHS, checkpoint_interval=CHECKPOINT_INTERVAL)

## 13. Save Trained Models

Save the trained generator and discriminator models for later use.

In [ ]:
# Configuration for saving models
SAVE_MODELS = True  # Set to False to skip saving
MODELS_SAVE_DIR = './saved_models'

def save_models(generator, discriminator, save_dir='./saved_models'):
    """
    Save the generator and discriminator models

    Args:
        generator: Generator model to save
        discriminator: Discriminator model to save
        save_dir: Directory to save models
    """
    import os

    # Create directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)

    # Define save paths
    generator_path = os.path.join(save_dir, 'generator.keras')
    discriminator_path = os.path.join(save_dir, 'discriminator.keras')

    # Alternative format (.h5)
    generator_path_h5 = os.path.join(save_dir, 'generator.h5')
    discriminator_path_h5 = os.path.join(save_dir, 'discriminator.h5')

    try:
        # Save in Keras format (.keras) - recommended
        generator.save(generator_path)
        discriminator.save(discriminator_path)

        # Also save in HDF5 format (.h5) for compatibility
        generator.save(generator_path_h5)
        discriminator.save(discriminator_path_h5)

        print("✓ Models saved successfully!")
        print(f"  - Generator (.keras): {generator_path}")
        print(f"  - Generator (.h5): {generator_path_h5}")
        print(f"  - Discriminator (.keras): {discriminator_path}")
        print(f"  - Discriminator (.h5): {discriminator_path_h5}")
        print(f"\nTo load these models later, use:")
        print(f"  LOAD_PRETRAINED_MODELS = True")
        print(f"  GENERATOR_MODEL_PATH = '{generator_path}'")
        print(f"  DISCRIMINATOR_MODEL_PATH = '{discriminator_path}'")

    except Exception as e:
        print(f"⚠ Error saving models: {e}")

# Save the models if training is complete
if SAVE_MODELS:
    print("Saving trained models...")
    save_models(generator, discriminator, MODELS_SAVE_DIR)
else:
    print("Model saving skipped (SAVE_MODELS = False)")

## 14. Visualize Training History

Plot the training losses over time.

In [ ]:
# Plot training history (NOW WITH PERCEPTUAL LOSS)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

epochs_range = range(1, len(history['gen_total_loss']) + 1)

# Generator Total Loss
axes[0, 0].plot(epochs_range, history['gen_total_loss'], 'b-', linewidth=2)
axes[0, 0].set_title('Generator Total Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# Generator GAN Loss
axes[0, 1].plot(epochs_range, history['gen_gan_loss'], 'g-', linewidth=2)
axes[0, 1].set_title('Generator Adversarial Loss', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].grid(True, alpha=0.3)

# Generator L1 Loss
axes[0, 2].plot(epochs_range, history['gen_l1_loss'], 'r-', linewidth=2)
axes[0, 2].set_title('Generator L1 Loss', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].grid(True, alpha=0.3)

# Generator Perceptual Loss (NEW)
axes[1, 0].plot(epochs_range, history['gen_percep_loss'], 'orange', linewidth=2)
axes[1, 0].set_title('Generator Perceptual Loss (NEW)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True, alpha=0.3)

# Discriminator Loss
axes[1, 1].plot(epochs_range, history['disc_loss'], 'm-', linewidth=2)
axes[1, 1].set_title('Discriminator Loss', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True, alpha=0.3)

# Combined Loss Comparison
axes[1, 2].plot(epochs_range, history['gen_gan_loss'], 'g-', linewidth=2, label='GAN Loss', alpha=0.7)
axes[1, 2].plot(epochs_range, history['gen_l1_loss'], 'r-', linewidth=2, label=f'L1 Loss (×{LAMBDA_L1})', alpha=0.7)
axes[1, 2].plot(epochs_range, history['gen_percep_loss'], 'orange', linewidth=2, label=f'Percep Loss (×{LAMBDA_PERCEPTUAL})', alpha=0.7)
axes[1, 2].set_title('Loss Components Comparison', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Loss')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history_improved.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training history visualized (with perceptual loss)")
print(f"  Final Generator Total Loss: {history['gen_total_loss'][-1]:.4f}")
print(f"  Final Generator GAN Loss: {history['gen_gan_loss'][-1]:.4f}")
print(f"  Final Generator L1 Loss: {history['gen_l1_loss'][-1]:.4f}")
print(f"  Final Generator Perceptual Loss: {history['gen_percep_loss'][-1]:.4f}")
print(f"  Final Discriminator Loss: {history['disc_loss'][-1]:.4f}")

## 15. Test the Trained Model

Test the generator on new degraded images.

In [ ]:
# Get multiple test batches for comprehensive testing
test_batches = []
for i, batch in enumerate(train_dataset.take(3)):
    test_batches.append(batch)

print(f"Testing on {len(test_batches)} batches")

# Test and visualize
for i, (test_deg, test_tar) in enumerate(test_batches):
    print(f"\nTest Batch {i + 1}:")
    generate_images(generator, test_deg, test_tar,
                   save_path=f'test_results_batch_{i+1}.png')

print("\n✓ Testing complete!")

## 16. Load and Use Saved Model

Demonstration of how to load and use the saved generator model.

In [25]:
# Load the saved generator
loaded_generator = keras.models.load_model(GENERATOR_SAVE_PATH)

print("✓ Generator model loaded successfully")

# Test the loaded model
test_deg, test_tar = next(iter(train_dataset))
generate_images(loaded_generator, test_deg, test_tar, save_path='loaded_model_test.png')

print("\n✓ Loaded model works correctly!")

## 17. Model Summary and Architecture Visualization

Summary of the complete GAN architecture.

In [ ]:
print("="*80)
print("GAN IMAGE RECONSTRUCTION MODEL - COMPLETE SUMMARY")
print("="*80)

print("\n📊 GENERATOR (U-Net Architecture)")
print("-" * 80)
print(f"  Architecture: U-Net with skip connections")
print(f"  Input shape: ({IMG_HEIGHT}, {IMG_WIDTH}, 3)")
print(f"  Output shape: ({IMG_HEIGHT}, {IMG_WIDTH}, 3)")
print(f"  Total parameters: {generator.count_params():,}")
print(f"  Trainable parameters: {sum([tf.size(v).numpy() for v in generator.trainable_variables]):,}")
print(f"\n  Encoder layers: 8 downsampling blocks (64 → 512 filters)")
print(f"  Bottleneck: 512 filters at 1x1 spatial resolution")
print(f"  Decoder layers: 7 upsampling blocks with skip connections")
print(f"  Output activation: tanh (range [-1, 1])")

print("\n📊 DISCRIMINATOR (PatchGAN)")
print("-" * 80)
print(f"  Architecture: PatchGAN Discriminator")
print(f"  Input 1: Degraded image ({IMG_HEIGHT}, {IMG_WIDTH}, 3)")
print(f"  Input 2: Target/Generated image ({IMG_HEIGHT}, {IMG_WIDTH}, 3)")
print(f"  Output shape: (30, 30, 1) - Patch predictions")
print(f"  Total parameters: {discriminator.count_params():,}")
print(f"  Trainable parameters: {sum([tf.size(v).numpy() for v in discriminator.trainable_variables]):,}")
print(f"\n  Purpose: Classify 30x30 patches as real or fake")
print(f"  This encourages high-frequency detail reconstruction")

print("\n📊 TRAINING CONFIGURATION")
print("-" * 80)
print(f"  Loss functions:")
print(f"    - Generator: Adversarial loss + L1 loss")
print(f"    - Discriminator: Binary cross-entropy")
print(f"  Optimizers:")
print(f"    - Generator: Adam (lr=2e-4, beta_1=0.5)")
print(f"    - Discriminator: Adam (lr=2e-4, beta_1=0.5)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image normalization: [-1, 1]")

print("\n📊 DATASET")
print("-" * 80)
print(f"  Degraded images: {degraded_dir}")
print(f"  Clean images: {clean_dir}")
print(f"  Total image pairs: {len(degraded_images)}")

print("\n" + "="*80)
print("✓ Model ready for image reconstruction tasks!")
print("="*80)